# Grid Rainout Retry — Convergence Resolution

Shows the outcome of `grid_retry.py` for every originally-failing (T, P) grid point.

**Retry passes:**
- **Pass A** — `accuracyChem = 1e-7` (tighter chemistry convergence)
- **Pass B** — `condUseSVD = True` (SVD solver for condensate system)
- **Pass C** — all iteration limits ×3 (10 000 gas, 100 000 joint)
- **Pass D** — `condIterChangeLimit = 50` (relax condensate step-size)
- **Pass E** — `nbSwitchToJoint = 0` (joint Newton from first combined iteration)

  *Rationale for Pass E:* The equilibrium condensation loop has a hardcoded stagnation
  detector that breaks the joint iteration at exactly **1500** steps
  (`checkpoint_interval=500 × max_stagnant_checkpoints=2`, with the first checkpoint
  always passing). The joint Newton solver — which couples gas+condensate simultaneously
  and breaks the gas/cond oscillation — only activates once
  `nb_combined_iter ≥ nb_switch_to_joint` (default 3000). Since 3000 > 1500, joint
  Newton never fires in the default configuration. Setting `nbSwitchToJoint=0`
  activates it immediately whenever active condensates are present.

A pass is only run if the previous pass did not fully resolve the profile.
Each point is assigned the **first** pass that fixed it (or `unsolved` if none did).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path

# ── Configuration ─────────────────────────────────────────────────────────────
OUTPUT_ROOT = Path("output/grid_day")

TEMPERATURES = [200, 400, 600, 800, 1000, 1200, 1400, 1600]  # K

COMPOSITIONS = {
    "EH3":         "EH3",
    "EL3":         "EL3",
    "R3":          "R3",
    "Krymka(LL3)": "Krymka(LL3)",
    "Bath(H4)":    "Bath(H4)",
}

PASSES = ["A", "B", "C", "D", "E"]

FIXED_COLS = {
    "grid_point", "iterations", "chem_iter", "cond_iter",
    "converged", "elem_conserved", "p_bar", "T_K",
    "ntot_cmneg3", "n_g_cmneg3", "m_u",
}

# Resolution categories in priority order
RESOLUTIONS = ["originally_ok"] + [f"pass{p}" for p in PASSES] + ["unsolved"]

RESOLUTION_COLORS = {
    "originally_ok": "#cccccc",  # light gray
    "passA":         "#2ca02c",  # green
    "passB":         "#98df8a",  # light green
    "passC":         "#1f77b4",  # blue
    "passD":         "#aec7e8",  # light blue
    "passE":         "#9467bd",  # purple  ← joint Newton fix
    "unsolved":      "#d62728",  # red
}

RESOLUTION_LABELS = {
    "originally_ok": "Originally OK",
    "passA":         "Fixed by Pass A (accuracyChem=1e-7)",
    "passB":         "Fixed by Pass B (condUseSVD)",
    "passC":         "Fixed by Pass C (more iterations)",
    "passD":         "Fixed by Pass D (condIterChangeLimit=50)",
    "passE":         "Fixed by Pass E (nbSwitchToJoint=0)",
    "unsolved":      "Still failing",
}

MS_OK = 8   # marker size for originally-OK points
MS_BAD = 18 # larger markers for failing/fixed points so they stand out

In [ ]:
def parse_monitor(path):
    """Parse a FastChem monitor.dat into a DataFrame (ok/fail → bool)."""
    with open(path) as f:
        header_line = f.readline().lstrip("#").strip()

    raw_cols = [c.strip() for c in header_line.split("\t") if c.strip()]
    cols = [
        c.replace("(", "_").replace(")", "")
         .replace("-", "neg").replace("<", "").replace(">", "")
        for c in raw_cols
    ]

    df = pd.read_csv(path, sep=r"\s+", comment="#", header=None, names=cols)

    bool_cols = [c for c in ["converged", "elem_conserved"] if c in df.columns]
    bool_cols += [c for c in cols[11:] if c in df.columns]
    for col in bool_cols:
        if df[col].dtype == object:
            df[col] = df[col].str.strip() == "ok"
        else:
            df[col] = df[col].astype(bool)

    return df


def point_is_failing(row):
    """True if this grid point failed convergence or element conservation."""
    if "converged" in row.index and not bool(row["converged"]):
        return True
    if "elem_conserved" in row.index and not bool(row["elem_conserved"]):
        return True
    return False


def monitor_is_failing(df):
    """Boolean Series: True where a row failed."""
    fail = pd.Series(False, index=df.index)
    if "converged" in df.columns:
        fail |= ~df["converged"].astype(bool)
    if "elem_conserved" in df.columns:
        fail |= ~df["elem_conserved"].astype(bool)
    return fail


def load_retry_resolution():
    """
    Build a flat DataFrame with one row per (composition, T_eq, grid_point).

    Columns:
      comp, T_eq, T_K, p_bar, originally_ok (bool), resolution (str)

    resolution is one of:
      'originally_ok' | 'passA' | 'passB' | 'passC' | 'passD' | 'unsolved'
    """
    records = []

    for label, slug in COMPOSITIONS.items():
        for T_eq in TEMPERATURES:
            chem_dir = OUTPUT_ROOT / f"{T_eq}K_{slug}_day" / "chemistry"
            orig_path = chem_dir / "monitor.dat"
            if not orig_path.exists():
                continue

            try:
                orig = parse_monitor(orig_path)
            except Exception as exc:
                print(f"  WARNING: cannot parse {orig_path}: {exc}")
                continue

            orig_fail = monitor_is_failing(orig)

            # Load retry pass monitors (only exist when original had failures)
            retry_dir = chem_dir / "retry"
            pass_fail = {}  # pass_label → boolean Series (True = still failing)
            for p in PASSES:
                pp = retry_dir / f"monitor_pass{p}.dat"
                if pp.exists():
                    try:
                        pdf = parse_monitor(pp)
                        pass_fail[p] = monitor_is_failing(pdf)
                    except Exception as exc:
                        print(f"  WARNING: cannot parse {pp}: {exc}")

            for i in range(len(orig)):
                row = orig.iloc[i]
                if not orig_fail.iloc[i]:
                    resolution = "originally_ok"
                else:
                    resolution = "unsolved"
                    for p in PASSES:
                        if p in pass_fail and not pass_fail[p].iloc[i]:
                            resolution = f"pass{p}"
                            break

                records.append({
                    "comp":         label,
                    "T_eq":         T_eq,
                    "T_K":          row["T_K"]   if "T_K"   in row.index else np.nan,
                    "p_bar":        row["p_bar"]  if "p_bar" in row.index else np.nan,
                    "originally_ok": not orig_fail.iloc[i],
                    "resolution":   resolution,
                })

    return pd.DataFrame(records)


print("Loading original and retry monitor files...")
df_all = load_retry_resolution()
print(f"Loaded {len(df_all):,} grid points across {df_all['comp'].nunique()} compositions.\n")

orig_fail  = df_all[~df_all["originally_ok"]]
n_orig_bad = len(orig_fail)
print(f"Originally failing points: {n_orig_bad}")
for res in RESOLUTIONS[1:]:   # skip 'originally_ok'
    n = (orig_fail["resolution"] == res).sum()
    print(f"  {RESOLUTION_LABELS[res]:45s}: {n}")

In [ ]:
# ── Summary table ──────────────────────────────────────────────────────────────
rows = []
for label in COMPOSITIONS:
    sub = df_all[df_all["comp"] == label]
    bad = sub[~sub["originally_ok"]]
    row = {"Composition": label, "Total pts": len(sub), "Originally failing": len(bad)}
    for p in PASSES:
        row[f"Fixed by {p}"] = (bad["resolution"] == f"pass{p}").sum()
    row["Still unsolved"] = (bad["resolution"] == "unsolved").sum()
    rows.append(row)

summary = pd.DataFrame(rows).set_index("Composition")
display(summary)

## 1. Resolution Map

One subplot per composition. Every (T, P) grid point is shown.  
**Gray** = originally OK.  Colored = originally failing; color indicates which pass fixed it (or red if still unsolved).  
Originally-failing points are drawn on top with larger markers so they are easy to spot.

In [ ]:
n = len(COMPOSITIONS)
ncols = min(3, n)
nrows = int(np.ceil(n / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(5.5 * ncols, 4.5 * nrows),
                          constrained_layout=True)
axes = np.array(axes).flatten()

for ax, label in zip(axes, COMPOSITIONS):
    sub = df_all[df_all["comp"] == label]

    # Draw originally-OK points first (background layer)
    ok  = sub[sub["resolution"] == "originally_ok"]
    ax.scatter(ok["T_K"], np.log10(ok["p_bar"]),
               c=RESOLUTION_COLORS["originally_ok"], s=MS_OK, linewidths=0, zorder=1)

    # Draw originally-failing points on top, colored by resolution
    bad = sub[sub["resolution"] != "originally_ok"]
    for res in RESOLUTIONS[1:]:   # passA … passD, unsolved
        pts = bad[bad["resolution"] == res]
        if pts.empty:
            continue
        ax.scatter(pts["T_K"], np.log10(pts["p_bar"]),
                   c=RESOLUTION_COLORS[res], s=MS_BAD, linewidths=0.4,
                   edgecolors="k", zorder=2)

    ax.set_xticks(TEMPERATURES)
    ax.set_xticklabels([str(t) for t in TEMPERATURES], rotation=45, ha="right")
    ax.set_xlabel("Temperature (K)")
    ax.set_ylabel("log₁₀ P (bar)")
    ax.invert_yaxis()

    n_bad     = len(bad)
    n_unsolved = (bad["resolution"] == "unsolved").sum()
    ax.set_title(f"{label}   [{n_bad} orig. failing, {n_unsolved} unsolved]", fontsize=10)

for ax in axes[n:]:
    ax.set_visible(False)

patches = [mpatches.Patch(color=RESOLUTION_COLORS[r], label=RESOLUTION_LABELS[r])
           for r in RESOLUTIONS if r in df_all["resolution"].values]
fig.legend(handles=patches, loc="lower right", fontsize=8, framealpha=0.9,
           title="Resolution status")
fig.suptitle("Retry resolution map — rainout grid", fontsize=13)
plt.show()

## 2. Per-Composition Resolution Breakdown

Stacked bar chart: for each composition, how many originally-failing points were resolved by each pass vs. remain unsolved.

In [ ]:
comp_labels = list(COMPOSITIONS.keys())
bar_cats = [f"pass{p}" for p in PASSES] + ["unsolved"]

counts = {
    cat: [
        ((df_all["comp"] == label) & (df_all["resolution"] == cat)).sum()
        for label in comp_labels
    ]
    for cat in bar_cats
}

x = np.arange(len(comp_labels))
fig, ax = plt.subplots(figsize=(7, 4), constrained_layout=True)

bottom = np.zeros(len(comp_labels))
for cat in bar_cats:
    vals = np.array(counts[cat])
    if vals.sum() == 0:
        continue
    ax.bar(x, vals, bottom=bottom,
           color=RESOLUTION_COLORS[cat], label=RESOLUTION_LABELS[cat],
           edgecolor="k", linewidth=0.4)
    # Annotate non-zero bars
    for xi, (v, b) in enumerate(zip(vals, bottom)):
        if v > 0:
            ax.text(xi, b + v / 2, str(v), ha="center", va="center",
                    fontsize=8, color="k")
    bottom += vals

ax.set_xticks(x)
ax.set_xticklabels(comp_labels, rotation=20, ha="right")
ax.set_ylabel("Grid points")
ax.set_title("Originally-failing points: resolution by pass", fontsize=11)
ax.legend(fontsize=8, loc="upper right")
plt.show()

## 3. Still-Unsolved Points

Detail table for every (T, P) point that remains failing after all retry passes.

In [ ]:
unsolved = df_all[df_all["resolution"] == "unsolved"].copy()

if unsolved.empty:
    print("All originally-failing points were resolved by at least one pass.")
else:
    print(f"{len(unsolved)} point(s) remain unsolved after all passes:\n")
    display(
        unsolved[["comp", "T_eq", "T_K", "p_bar"]]
        .sort_values(["comp", "T_K", "p_bar"])
        .reset_index(drop=True)
    )

## 4. Iteration Count After Best Fix

For each originally-failing point, shows the iteration count from the **first pass that fixed it** (or from Pass D if still unsolved and Pass D ran).  
High counts even after fixing indicate the solver was still struggling.

In [ ]:
def load_best_pass_iters():
    """
    For each originally-failing point, return the iteration count
    from the first pass that fixed it (or the last available pass if unsolved).
    """
    records = []

    for label, slug in COMPOSITIONS.items():
        for T_eq in TEMPERATURES:
            chem_dir = OUTPUT_ROOT / f"{T_eq}K_{slug}_day" / "chemistry"
            orig_path = chem_dir / "monitor.dat"
            if not orig_path.exists():
                continue

            try:
                orig = parse_monitor(orig_path)
            except Exception:
                continue

            orig_fail = monitor_is_failing(orig)
            if not orig_fail.any():
                continue  # no failures in this profile

            retry_dir = chem_dir / "retry"
            pass_dfs = {}
            for p in PASSES:
                pp = retry_dir / f"monitor_pass{p}.dat"
                if pp.exists():
                    try:
                        pass_dfs[p] = parse_monitor(pp)
                    except Exception:
                        pass

            for i in range(len(orig)):
                if not orig_fail.iloc[i]:
                    continue

                best_iters = np.nan
                best_pass  = "unsolved"
                last_iters = np.nan

                for p in PASSES:
                    if p not in pass_dfs:
                        continue
                    prow = pass_dfs[p].iloc[i]
                    last_iters = prow["iterations"] if "iterations" in prow.index else np.nan
                    if not monitor_is_failing(pass_dfs[p]).iloc[i]:
                        best_iters = last_iters
                        best_pass  = f"pass{p}"
                        break

                iters = best_iters if not np.isnan(best_iters) else last_iters

                records.append({
                    "comp":      label,
                    "T_K":       orig.iloc[i]["T_K"],
                    "p_bar":     orig.iloc[i]["p_bar"],
                    "resolution": best_pass,
                    "iterations": iters,
                })

    return pd.DataFrame(records)


df_iters = load_best_pass_iters()

if df_iters.empty:
    print("No originally-failing points found (nothing to plot).")
else:
    iter_vmax = np.nanpercentile(df_iters["iterations"].dropna(), 99)

    n = len(COMPOSITIONS)
    ncols = min(3, n)
    nrows = int(np.ceil(n / ncols))

    fig, axes = plt.subplots(nrows, ncols, figsize=(5.5 * ncols, 4.5 * nrows),
                              constrained_layout=True)
    axes = np.array(axes).flatten()
    sc = None

    for ax, label in zip(axes, COMPOSITIONS):
        sub = df_iters[df_iters["comp"] == label].dropna(subset=["iterations"])
        if sub.empty:
            ax.set_title(f"{label}   [no data]", fontsize=10)
            continue

        sc = ax.scatter(
            sub["T_K"], np.log10(sub["p_bar"]),
            c=sub["iterations"], cmap="YlOrRd",
            vmin=0, vmax=iter_vmax,
            s=MS_BAD, linewidths=0.4, edgecolors="k", zorder=2,
        )
        ax.set_xticks(TEMPERATURES)
        ax.set_xticklabels([str(t) for t in TEMPERATURES], rotation=45, ha="right")
        ax.set_xlabel("Temperature (K)")
        ax.set_ylabel("log₁₀ P (bar)")
        ax.invert_yaxis()
        ax.set_title(f"{label}   [max iter = {int(sub['iterations'].max())}]", fontsize=10)

    for ax in axes[n:]:
        ax.set_visible(False)

    if sc is not None:
        fig.colorbar(sc, ax=axes[:n], label="iterations (best-fix pass)",
                     shrink=0.6, pad=0.02)
    fig.suptitle("Iteration count at best-fix pass — originally-failing points", fontsize=13)
    plt.show()